Beam-search

- https://habr.com/ru/articles/599673/
- https://habr.com/ru/articles/745314/
- https://habr.com/ru/articles/346578/
- https://habr.com/ru/companies/vk/articles/579412/
- https://habr.com/ru/companies/sberdevices/articles/666420/

Beam Search — это эвристический алгоритм поиска, широко применяемый в задачах обработки естественного языка (Natural Language Processing, NLP). Основная цель этого метода заключается в поиске оптимальной последовательности токенов (слов или символов) в условиях огромного количества возможных вариантов. Давайте рассмотрим подробнее, как Beam Search применяется в контексте NLP.
Что такое Beam Search?

Beam Search — это способ поиска, основанный на эвристическом подходе, который пытается найти лучшую последовательность элементов (например, слов в предложении) путём рассмотрения ограниченного числа возможных путей (или "лучей") на каждом этапе. Идея состоит в том, чтобы сосредоточиться лишь на небольшом количестве наиболее перспективных вариантов, игнорируя остальные. Этот метод существенно сокращает пространство поиска, делая возможным эффективное нахождение хороших решений в ситуациях, когда полное переборное исследование всех возможных вариантов невозможно.
Как работает Beam Search в NLP?

Представим себе задачу автоматического формирования текста или машинного перевода. Задача состоит в том, чтобы на основе некоторого начального ввода (например, "начало предложения") найти наилучшую последовательность слов, образующих осмысленное предложение. Рассмотрим пошагово, как это происходит с применением Beam Search:

- Инициализация: Начинаем с начального состояния (например, пустого ввода или первого слова).
- Генерация гипотез: На каждом шаге мы генерируем возможные продолжения для текущего состояния. В случае языковой модели это могут быть следующие слова, которые могут быть использованы в следующем токене.
- Оценка гипотез: Каждое возможное продолжение оценивается по некоторой метрике (например, по вероятности того, что именно это слово должно идти дальше согласно модели).
- Выбор лучших гипотез: Из всех возможных продолжений выбирается ограниченное количество лучших (обычно это параметр beam_width, задающий ширину пучка). Остальные гипотезы отбрасываются.
- Рекурсия: Процесс повторяется для каждого выбранного продолжения до тех пор, пока не будет достигнут конец предложения или другая заранее определённая точка остановки.
- Окончание: Когда поиск завершён, возвращаются лучшие последовательности, найденные методом Beam Search.

Плюсы и минусы Beam Search в NLP
Плюсы:

- Эффективность: Позволяет находить хорошие решения в больших пространствах состояний без полного перебора.
- Контролируемая сложность: Параметр beam_width даёт возможность настраивать компромисс между точностью и производительностью.
- Подходит для сложных задач: Используется в широком спектре приложений, включая машинный перевод, автоматическое формулирование текста и др.

Минусы:

- Не гарантирует оптимальное решение: Может пропустить лучшие варианты, если они находятся вне текущего пучка.
- Чувствителен к параметрам: Неправильно подобранная ширина пучка может привести либо к пропуску хороших решений, либо к чрезмерному увеличению времени вычисления.
- Требует настройки: Подбор правильных параметров (ширина пучка, длина последовательности и т.п.) требует экспериментов и тестов.

## Задача
1. Собрать полный пайплан по генерации текста с примением процедуры препроцессинга, токенизации и генерации используя алгоритм beam search. Попытаться добитьсякачественной генерации текста. За основу взять лабораторную №3. Попытаться реализовать как можно болеьше предложенных методов.   

In [1]:
import nltk
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, LayerNormalization, Bidirectional
import nltk
from nltk.tokenize import word_tokenize
import re
import numpy as np

nltk.download('punkt_tab')
with open('marsianskie_hroniki.txt', 'r', encoding='utf-8') as file:
    text = file.read()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
# Обработка текста
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^а-я0-9 ]', ' ', text)
    tokens = word_tokenize(text)
    return tokens
tokens = preprocess_text(text)
tokens[:10]

['моей',
 'жене',
 'маргарет',
 'с',
 'искренней',
 'любовью',
 'великое',
 'дело',
 'способность',
 'удивляться']

In [3]:
# Создаём словарь уникальных токенов и сопоставление слово-индекс
vocab = sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
len(vocab)

14465

In [4]:
# Преобразуем текст в последовательность индексов
text_indices = [word2idx[w] for w in tokens]

# Создаём обучающие последовательности
seq_length = 10
sequences = []
next_words = []
for i in range(len(text_indices) - seq_length):
    sequences.append(text_indices[i:i+seq_length])
    next_words.append(text_indices[i+seq_length])

X = np.array(sequences)
y = np.array(next_words)
X.shape, y.shape

((54451, 10), (54451,))

In [5]:
# Модель LSTM
model = Sequential()
model.add(Embedding(input_dim=len(vocab), output_dim=256))
model.add(Bidirectional(LSTM(256, return_sequences=True)))
model.add(Dropout(0.2))
model.add(LayerNormalization())
model.add(LSTM(256))
model.add(Dropout(0.2))
model.add(Dense(len(vocab), activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
model.build(input_shape=(None, seq_length))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 10, 256)        │     3,703,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 10, 512)        │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization             │ (None, 10, 512)        │         1,024 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       787,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 14465)          │     3,717,505 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,259,649 (35.32 MB)

 Trainable params: 9,259,649 (35.32 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(X, y, batch_size=256, epochs=50)

Epoch 1/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - loss: 8.5953
Epoch 2/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 7.8529
Epoch 3/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 7.6790
Epoch 4/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 7.4974
Epoch 5/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 7.2381
Epoch 6/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 6.9985
Epoch 7/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 6.7491
Epoch 8/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 6.4831
Epoch 9/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 6.2158
Epoch 10/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 5.9742
Epoch 11/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 5.7510
Epoch 12/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 5.5262
Epoch 13/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 5.2826
Epoch 14/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 5.0392
Epoch 15/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 5

In [66]:
def beam_search(model, seed_text, word2idx, idx2word, seq_length, beam_width, max_gen_len):
    tokens = preprocess_text(seed_text)
    init_seq = [word2idx.get(w, 0) for w in tokens]
    hypotheses = [(init_seq, 0.0)]
    for _ in range(max_gen_len):
        all_candidates = []
        for seq, score in hypotheses:
            input_seq = seq[-seq_length:]
            input_padded = pad_sequences([input_seq], maxlen=seq_length, padding='pre')
            preds = model.predict(input_padded, verbose=0)[0]
            log_probs = np.log(preds + 1e-8)
            top_ids = np.argsort(log_probs)[-beam_width:]
            for idx in top_ids:
                new_seq = seq + [int(idx)]
                new_score = score + float(log_probs[idx])
                all_candidates.append((new_seq, new_score))
        ordered = sorted(all_candidates, key=lambda x: x[1], reverse=True)
        hypotheses = ordered[:beam_width]
    best_seq, best_score = hypotheses[0]
    generated_words = [idx2word.get(i, '') for i in best_seq]
    return ' '.join(generated_words), best_score

In [67]:
seed_text = 'апрель на марсе шел'
beam_width = 5
max_gen_len = 4

generated, _ = beam_search(model, seed_text, word2idx, idx2word, seq_length=seq_length,
                           beam_width=beam_width, max_gen_len=max_gen_len)
generated

'апрель на марсе шел из дома и пропал'

In [68]:
seed_text = 'он открыл лавку на марсе'
beam_width = 5
max_gen_len = 4

generated, _ = beam_search(model, seed_text, word2idx, idx2word, seq_length=seq_length,
                           beam_width=beam_width, max_gen_len=max_gen_len)
generated

'он открыл лавку на марсе и по прежнему одинокий'

In [69]:
seed_text = 'далеко от америки'
beam_width = 5
max_gen_len = 6

generated, _ = beam_search(model, seed_text, word2idx, idx2word, seq_length=seq_length,
                           beam_width=beam_width, max_gen_len=max_gen_len)
generated

'далеко от америки он покрикивал на дне мертвого моря'

In [70]:
seed_text = 'во мраке сидел мужчина'
beam_width = 5
max_gen_len = 9

generated, _ = beam_search(model, seed_text, word2idx, idx2word, seq_length=seq_length,
                           beam_width=beam_width, max_gen_len=max_gen_len)
generated

'во мраке сидел мужчина в той стороне дома стоял резкий запах потного тела'

In [ ]:
seed_text = 'во мраке сидел мужчина'
beam_width = 5

for max_gen in range(1, 20):
    max_gen_len = max_gen
    generated, score = beam_search(model, seed_text, word2idx, idx2word, seq_length=seq_length,
                                   beam_width=beam_width, max_gen_len=max_gen_len)
    print(f'Сгенерировано токенов: {max_gen} / Логит-скор: {score}\n{generated}\n')

Сгенерировано токенов: 1 / Логит-скор: -0.6913985013961792
во мраке сидел мужчина и

Сгенерировано токенов: 2 / Логит-скор: -1.906578540802002
во мраке сидел мужчина и еще

Сгенерировано токенов: 3 / Логит-скор: -3.068937838077545
во мраке сидел мужчина в той стороне

Сгенерировано токенов: 4 / Логит-скор: -3.5988730788230896
во мраке сидел мужчина в той стороне дома

Сгенерировано токенов: 5 / Логит-скор: -4.755474475445226
во мраке сидел мужчина в той неделе и за

Сгенерировано токенов: 6 / Логит-скор: -6.090434876503423
во мраке сидел мужчина в той неделе и его упала

Сгенерировано токенов: 7 / Логит-скор: -6.597442019730806
во мраке сидел мужчина в той стороне дома во сне поднялась

Сгенерировано токенов: 8 / Логит-скор: -6.607709535397589
во мраке сидел мужчина в той стороне дома во сне поднялась и

Сгенерировано токенов: 9 / Логит-скор: -6.835503242909908
во мраке сидел мужчина в той стороне дома стоял резкий запах потного тела

Сгенерировано токенов: 10 / Логит-скор: -8.43754544

Дополнительная задача, на дополнительный бал:

Попытаться реализовать

Exhaustive Search

Exhaustive Search проверяет все возможные варианты, гарантируя нахождение абсолютно лучшего решения. Однако, это непрактично для большинства реальных задач из-за огромной вычислительной сложности.



In [9]:
# Здесь был код...

*Нужно ли что-то исправить?*

&check; Пхе &#x2610;